# LR

In [1]:
!pip install scikit-learn scipy joblib

Looking in indexes: http://mirrors.aliyun.com/pypi/simple


In [2]:
import pandas as pd
import numpy as np
import os

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

In [3]:
DATA_PATH = "FeatureC_Repeated"
OUTPUT_PATH = "LR_FeatureC_Results"

os.makedirs(OUTPUT_PATH, exist_ok=True)

N_REPEATS = 10

C_VALUES = [0.1, 1, 10]

N_INNER_REPEATS = 3
VALID_RATIO = 0.2
BASE_SEED = 42

In [4]:
def get_feature_cols(df):
    return [
        col for col in df.columns
        if col not in ["userId", "movieId", "label"]
    ]

In [5]:
def stratified_split_from_scratch(df, label_col, test_ratio=0.2, random_seed=42):
    rng = np.random.default_rng(random_seed)
    train_indices = []
    test_indices = []
    for label_value in df[label_col].unique():
        label_indices = df[df[label_col] == label_value].index.to_numpy()
        rng.shuffle(label_indices)

        test_size = int(len(label_indices) * test_ratio)

        test_indices.extend(label_indices[:test_size])
        train_indices.extend(label_indices[test_size:])

    train_df = df.loc[train_indices].sample(
        frac=1,
        random_state=random_seed
    ).reset_index(drop=True)

    test_df = df.loc[test_indices].sample(
        frac=1,
        random_state=random_seed
    ).reset_index(drop=True)

    return train_df, test_df

In [6]:
def compute_basic_metrics_from_scratch(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    tp = np.sum((y_true == 1) & (y_pred == 1))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))

    accuracy = (tp + tn) / len(y_true)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0

    recall = tp / (tp + fn) if (tp + fn) > 0 else 0

    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0
        else 0
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn
    }

In [7]:
def compute_auc_from_scratch(y_true, y_prob):
    y_true = np.array(y_true)
    y_prob = np.array(y_prob)

    sorted_indices = np.argsort(-y_prob)
    y_true_sorted = y_true[sorted_indices]

    pos_count = np.sum(y_true == 1)
    neg_count = np.sum(y_true == 0)

    if pos_count == 0 or neg_count == 0:
        return 0

    tp = 0
    fp = 0

    tpr_list = [0]
    fpr_list = [0]

    for label in y_true_sorted:
        if label == 1:
            tp += 1
        else:
            fp += 1

        tpr_list.append(tp / pos_count)
        fpr_list.append(fp / neg_count)

    auc = 0

    for i in range(1, len(tpr_list)):
        auc += (
            (fpr_list[i] - fpr_list[i - 1])
            *
            (tpr_list[i] + tpr_list[i - 1])
            / 2
        )

    return auc

In [8]:
def train_and_evaluate_lr(train_df, test_df, C_value):
    feature_cols = get_feature_cols(train_df)

    X_train = train_df[feature_cols]
    y_train = train_df["label"]

    X_test = test_df[feature_cols]
    y_test = test_df["label"]

    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    model = LogisticRegression(
        C=C_value,
        penalty="l2",
        solver="liblinear",
        max_iter=1000,
        random_state=42
    )

    model.fit(X_train_scaled, y_train)

    y_pred = model.predict(X_test_scaled)
    y_prob = model.predict_proba(X_test_scaled)[:, 1]

    metrics = compute_basic_metrics_from_scratch(y_test, y_pred)
    metrics["auc"] = compute_auc_from_scratch(y_test, y_prob)

    return metrics

In [9]:
def tune_lr_C_from_scratch(train_df, C_values, n_inner_repeats=3, valid_ratio=0.2, base_seed=100):
    tuning_records = []

    for C_value in C_values:
        inner_f1_scores = []

        for inner_id in range(n_inner_repeats):
            inner_train_df, valid_df = stratified_split_from_scratch(
                train_df,
                label_col="label",
                test_ratio=valid_ratio,
                random_seed=base_seed + inner_id
            )

            metrics = train_and_evaluate_lr(
                train_df=inner_train_df,
                test_df=valid_df,
                C_value=C_value
            )

            inner_f1_scores.append(metrics["f1"])

        tuning_records.append({
            "C": C_value,
            "mean_validation_f1": np.mean(inner_f1_scores),
            "std_validation_f1": np.std(inner_f1_scores, ddof=1)
        })

    tuning_df = pd.DataFrame(tuning_records)

    best_C = tuning_df.sort_values(
        by="mean_validation_f1",
        ascending=False
    ).iloc[0]["C"]

    return best_C, tuning_df

In [10]:
all_results = []
all_tuning_results = []

for repeat_id in range(1, N_REPEATS + 1):

    print("=" * 60)
    print(f"Outer Repeat {repeat_id:02d}")
    print("=" * 60)

    repeat_folder = os.path.join(
        DATA_PATH,
        f"repeat_{repeat_id:02d}"
    )

    train_path = os.path.join(repeat_folder, "feature_C_train.csv")
    test_path = os.path.join(repeat_folder, "feature_C_test.csv")

    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)

    print("Train shape:", train_df.shape)
    print("Test shape:", test_df.shape)

    best_C, tuning_df = tune_lr_C_from_scratch(
        train_df=train_df,
        C_values=C_VALUES,
        n_inner_repeats=N_INNER_REPEATS,
        valid_ratio=VALID_RATIO,
        base_seed=1000 + repeat_id * 10
    )

    print("Best C:", best_C)

    tuning_df["outer_repeat"] = repeat_id
    all_tuning_results.append(tuning_df)

    final_metrics = train_and_evaluate_lr(
        train_df=train_df,
        test_df=test_df,
        C_value=best_C
    )

    result_row = {
        "outer_repeat": repeat_id,
        "best_C": best_C,
        **final_metrics
    }

    all_results.append(result_row)

    print("Accuracy :", round(final_metrics["accuracy"], 4))
    print("Precision:", round(final_metrics["precision"], 4))
    print("Recall   :", round(final_metrics["recall"], 4))
    print("F1       :", round(final_metrics["f1"], 4))
    print("AUC      :", round(final_metrics["auc"], 4))

Outer Repeat 01
Train shape: (160000, 253)
Test shape: (39999, 253)
Best C: 10.0
Accuracy : 0.6381
Precision: 0.6485
Recall   : 0.6026
F1       : 0.6247
AUC      : 0.6739
Outer Repeat 02
Train shape: (160000, 253)
Test shape: (39999, 253)
Best C: 10.0
Accuracy : 0.6357
Precision: 0.6418
Recall   : 0.6135
F1       : 0.6273
AUC      : 0.6709
Outer Repeat 03
Train shape: (160000, 253)
Test shape: (39999, 253)
Best C: 10.0
Accuracy : 0.6365
Precision: 0.6438
Recall   : 0.6106
F1       : 0.6267
AUC      : 0.6731
Outer Repeat 04
Train shape: (160000, 253)
Test shape: (39999, 253)
Best C: 10.0
Accuracy : 0.6339
Precision: 0.6452
Recall   : 0.5941
F1       : 0.6186
AUC      : 0.6722
Outer Repeat 05
Train shape: (160000, 253)
Test shape: (39999, 253)
Best C: 1.0
Accuracy : 0.6346
Precision: 0.6432
Recall   : 0.604
F1       : 0.623
AUC      : 0.6719
Outer Repeat 06
Train shape: (160000, 253)
Test shape: (39999, 253)
Best C: 1.0
Accuracy : 0.6398
Precision: 0.6477
Recall   : 0.6123
F1       : 0.6

In [11]:
results_df = pd.DataFrame(all_results)
tuning_results_df = pd.concat(all_tuning_results, ignore_index=True)

results_path = os.path.join(OUTPUT_PATH, "LR_FeatureC_repeated_results.csv")
tuning_path = os.path.join(OUTPUT_PATH, "LR_FeatureC_tuning_results.csv")

results_df.to_csv(results_path, index=False, encoding="utf-8-sig")
tuning_results_df.to_csv(tuning_path, index=False, encoding="utf-8-sig")

print("Saved repeated test results to:")
print(results_path)

print("Saved tuning results to:")
print(tuning_path)

results_df

Saved repeated test results to:
LR_FeatureC_Results/LR_FeatureC_repeated_results.csv
Saved tuning results to:
LR_FeatureC_Results/LR_FeatureC_tuning_results.csv


,outer_repeat,best_C,accuracy,precision,recall,f1,tp,tn,fp,fn,auc
0,1,10.0,0.638141,0.648471,0.602601,0.624695,12046,13479,6530,7944,0.673945
1,2,10.0,0.635716,0.641792,0.613507,0.627331,12264,13164,6845,7726,0.670852
2,3,10.0,0.636541,0.643792,0.610555,0.626733,12205,13256,6753,7785,0.673109
3,4,10.0,0.633866,0.645174,0.594147,0.618610,11877,13477,6532,8113,0.672184
4,5,1.0,0.634616,0.643158,0.604002,0.622965,12074,13310,6699,7916,0.671938
5,6,1.0,0.639816,0.647737,0.612256,0.629497,12239,13353,6656,7751,0.676813
6,7,1.0,0.640466,0.646733,0.618359,0.632228,12361,13257,6752,7629,0.677412
7,8,10.0,0.636141,0.642363,0.613507,0.627604,12264,13181,6828,7726,0.673367
8,9,1.0,0.638741,0.646205,0.612456,0.628878,12243,13306,6703,7747,0.675147
9,10,10.0,0.640691,0.647873,0.615658,0.631355,12307,13320,6689,7683,0.676190


In [12]:
summary_records = []

for metric in ["accuracy", "precision", "recall", "f1", "auc"]:
    values = results_df[metric].values

    summary_records.append({
        "metric": metric,
        "mean": np.mean(values),
        "std": np.std(values, ddof=1),
        "standard_error": np.std(values, ddof=1) / np.sqrt(len(values))
    })

summary_df = pd.DataFrame(summary_records)

summary_path = os.path.join(OUTPUT_PATH, "LR_FeatureC_summary.csv")
summary_df.to_csv(summary_path, index=False, encoding="utf-8-sig")

summary_df

,metric,mean,std,standard_error
0,accuracy,0.637473,0.002444,0.000773
1,precision,0.645330,0.002433,0.000770
2,recall,0.609705,0.007296,0.002307
3,f1,0.626990,0.004063,0.001285
4,auc,0.674096,0.002217,0.000701


In [13]:
best_C_frequency = results_df["best_C"].value_counts().reset_index()
best_C_frequency.columns = ["C", "frequency"]

best_C_frequency_path = os.path.join(OUTPUT_PATH, "LR_FeatureC_best_C_frequency.csv")
best_C_frequency.to_csv(best_C_frequency_path, index=False, encoding="utf-8-sig")

best_C_frequency

,C,frequency
0,10.0,6
1,1.0,4
